# **PIPELINE NOTES:**

> #### **UPDATES**
> - 08/12/2025: SR made updates to fix naming issues if mask is not "cell", add mask object count QC function, and correct saving errors for XY_bins and XY_wedges images.
> - 08/04/2025: SR created pipeline as described below.

### **Created 08/04/2025 by SR using v0.8 code from ZSC and v1.0.0 NDCN/infer-subc code**
This pipeline was built for MCZ on 08/04/2025 by SR so that MCZ could analyze the iPSC-->iNeuron data she needs to include in her paper. The based of each function below is from SCohenLab/infer-subc v1.0.0. Additional have been made to add the following features from code ZSC has updated in preparation for v2.0.0:
1. multi-way interactions analysis
2. optional use of each analysis type

This analysis should be run using NDCN/infer-subc that is installed using "pip install infer-subc"


# **Import**

Run all cells here to load the functions necessary for your analysis.

In [1]:
import numpy as np
import pandas as pd
import time
import itertools 
import warnings
import string


from pathlib import Path
from typing import Union, List
from infer_subc.core.img import apply_mask
from skimage.measure import regionprops_table, regionprops, label
from infer_subc.utils.batch import list_image_files, find_segmentation_tiff_files
from infer_subc.core.file_io import read_czi_image, read_tiff_image, export_inferred_organelle
#from infer_subc.utils.stats_helpers import (assert_uint16_labels)
from infer_subc.utils.stats import (_assert_uint16_labels, 
                                    surface_area_from_props, 
                                    get_XY_distribution, 
                                    get_Z_distribution, 
                                    get_region_morphology_3D)

pd.set_option('display.max_columns', None)

In [ ]:
#### FUNCTIONS THAT NEEDED EDITING FOR BATCH_PROCESS_QUANTIFICATION FUNCTION ####
#################################################################################

# from ZSC for implementation in future infer-subc v2.0.0
def make_dict_20250804(list_obj_names: list[str],
               list_obj_segs: list[np.ndarray]):
    organelle_segs = {}                                                     
    for idx, name in enumerate(list_obj_names):                                  
        if name == 'ER':                                                    
            organelle_segs[name]=(list_obj_segs[idx]>0).astype(np.uint16)        
        else:                                                       
            organelle_segs[name]=list_obj_segs[idx]
    return organelle_segs



# from ZSC for implementation in future infer-subc v2.0.0
def create_overlap_20250804(orgs:str,
                   organelle_segs: dict[str:np.ndarray],
                   splitter: str="X") -> tuple[np.ndarray, np.ndarray]: 
    ##########################################
    ## CREATE OVERLAP
    ##########################################
    site = np.ones_like(organelle_segs[orgs.split(splitter)[0]]) 
    for org in orgs.split(splitter):        
        b = organelle_segs[org]             # select organelle
        valid = (b>0)*(site>0)              # logical and: select MASK of region where site (everything or previous overlap) overlaps with org b
        digit = len(str(np.max(site)))      # find the max ID number in the site image
        site = (b*(10**(digit)))+site       # multiply the organelle b image by 10^digit (up to the next highest multiplier of 10 compared to the original org number per image) then add the site image to that --> preservation of declumped objects
        site[valid.astype(bool)==False]=0   # select everywhere outside the overlap in the site image and get rid of it --> only object multiplied by 10^digit, but now there is separation between declumped neighbors inherited from the original ID numbers added in
        site = label(site)                  # label the overlap to minimize the max number & provide logical number system to contacts in end
    return site



# from ZSC for implementation in future infer-subc v2.0.0
def interaction_metric_analysis_20250804(overlap_ID: str,
                                list_obj_names: list[str],
                                list_obj_segs: list[np.ndarray],
                                intensity_img: np.ndarray, 
                                channel_axis: int,
                                mask: np.ndarray,
                                mask_name: str,
                                # regions_dict: dict[str:np.ndarray],
                                splitter: str="X",
                                scale: Union[tuple, None]=None,
                                include_dist:bool=False, 
                                dist_centering_obj: Union[np.ndarray, None]=None,
                                dist_num_bins: Union[int, None]=None,
                                dist_zernike_degrees: Union[int, None]=None,
                                dist_center_on: Union[bool, None]=None,
                                dist_keep_center_as_bin: Union[bool, None]=None,
                                return_site: bool=False):
    """
    collect volumentric measurements of intersection between n organelle types

    Parameters
    ------------
    overlap_ID: str
        a value used to describe the organelles present in the overlap that can be divided by the splitter value
    org_dict: dict
        a dictionary of all object segmentations assigned to keys with their objects
    mask: np.ndarray
        3D (ZYX) binary mask of the area to measure interactions from
    splitter: str
        a value used to separate the overlap_ID to determine objects present in overlap
    scale: tuple
        a value present in the metadata determining the scale of the (ZYX) axis
    include_dist:bool=False
        *optional*
        True = include the XY and Z distribution measurements of the overlap sites within the masked region 
        (utilizing the functions get_XY_distribution() and get_Z_distribution() from Infer-subc)
        False = do not include distirbution measurements
    dist_centering_obj: Union[np.ndarray, None]=None
        ONLY NEEDED IF include_dist=True; if None, the center of the mask will be used
        3D (ZYX) np.ndarray containing the object to use for centering the XY distribution mask
    dist_num_bins: Union[int, None]=None
        ONLY NEEDED IF include_dist=True; if None, the default is 5
    dist_zernike_degrees: Unions[int, None]=None,
        ONLY NEEDED IF include_dist=True; if None, the zernike share measurements will not be included in the distribution
        the number of zernike degrees to include for the zernike shape descriptors
    dist_center_on: Union[bool, None]=None
        ONLY NEEDED IF include_dist=True; if None, the default is False
        True = distribute the bins from the center of the centering object
        False = distribute the bins from the edge of the centering object
    dist_keep_center_as_bin: Union[bool, None]=None
        ONLY NEEDED IF include_dist=True; if None, the default is True
        True = include the centering object area when creating the bins
        False = do not include the centering object area when creating the bins


    Regionprops measurements:
    ------------------------
    ['label',
    'centroid',
    'bbox',
    'area',
    'equivalent_diameter',
    'extent',
    'feret_diameter_max',
    'euler_number',
    'convex_area',
    'solidity',
    'axis_major_length',
    'axis_minor_length']

    Additional measurements:
    ----------------------
    ['surface_area']

    
    Returns
    -------------
    pandas dataframe of containing regionprops measurements (columns) for each overlap region (rows)
    
    """
    #########################
    ## CREATE ORG_DICT
    #########################
    org_dict = make_dict_20250804(list_obj_names, list_obj_segs)


    #########################
    ## CREATE OVERLAP REGIONS
    #########################
    # run create overlap function
    site = create_overlap_20250804(overlap_ID, org_dict, splitter)

    #############################################################################################
    #assert the nth order overlap to within the cellmask
    labels = label(apply_mask(site, mask)).astype(int)


    ##########################################
    ## CREATE LIST OF REGIONPROPS MEASUREMENTS
    ##########################################
    # dealing with numerous solidity warning from regionprops
    warnings.simplefilter("ignore")

    # start with LABEL
    properties = ["label"]

    # add position
    properties += ["centroid", "bbox"]

    # add area
    properties += ["area", "equivalent_diameter"] # "num_pixels", 

    # add shape measurements - NOTE: can't include minor axis measure because some of the contact sites are only one pixel
    properties += ["extent", "euler_number", "solidity", "axis_major_length", "slice"] # "feret_diameter_max",  , "axis_minor_length"
    
    properties = properties + ["min_intensity", "max_intensity", "mean_intensity"]

    #######################
    ## ADD EXTRA PROPERTIES
    #######################
    def standard_deviation_intensity(region, intensities):
        return np.std(intensities[region])

    extra_properties = [standard_deviation_intensity]

    ##################
    ## RUN REGIONPROPS
    ##################
    if channel_axis == len(scale):
        intensity_input = intensity_img
    else:
        intensity_input = np.moveaxis(intensity_img, channel_axis, -1)

    props = regionprops_table(labels, 
                              intensity_image=intensity_input, 
                              properties=properties, 
                              extra_properties=extra_properties, 
                              spacing=scale)

    ##################################################################
    ## RUN SURFACE AREA FUNCTION SEPARATELY AND APPEND THE PROPS_TABLE
    ##################################################################
    surface_area_tab = pd.DataFrame(surface_area_from_props(labels, props, scale))

    #################################################################################################


    ########################################################
    ## LIST WHICH ORGANELLES ARE INVOLVED IN THE INTERACTION
    ########################################################
    over_inv = []
    involved = overlap_ID.split(splitter)
    indexes = {overlap_ID: []}

    # cells = cell_finder_pt5(scale=scale, obj=labels, mask=mask, mask_name=mask_name, props=props)
    # regions = region_finder_pt5(scale=scale, obj=labels, regions=regions_dict, props=props)

    for index, l in enumerate(props["label"]):
        over_inv.clear()
        for org in involved:
            volume = labels[props["slice"][index]]
            lorg = org_dict[org][props["slice"][index]]
            volume = volume==l
            lorg = lorg[volume]                                 
            all_inv = np.unique(lorg[lorg>0]).tolist()          
            if len(all_inv) != 1:
                print(f"we have an error.  as-> {all_inv}") # ensure that only one org object is 
            over_inv.append(f"{all_inv[0]}")
        indexes[overlap_ID].append('_'.join(over_inv))

        
    ##################################################
    ## CREATE COMBINED DATAFRAME OF THE QUANTIFICATION
    ##################################################
    props_table = pd.DataFrame(props)
    props_table.rename(columns={'label': 'idx'}, inplace=True)
    props_table.drop(columns=['slice'], inplace=True)
    props_table.insert(0, 'label',value=indexes[overlap_ID])
    props_table.insert(0, "object", overlap_ID)
    props_table.rename(columns={"area": "volume"}, inplace=True)
    props_table.insert(13, "surface_area", surface_area_tab)
    props_table.insert(14, "SA_to_volume_ratio", props_table["surface_area"].div(props_table["volume"]))
    if scale is not None:
        round_scale = (round(scale[0], 4), round(scale[1], 4), round(scale[2], 4))
        props_table.insert(loc=2, column="scale", value=f"{round_scale}")
    else: 
        props_table.insert(loc=2, column="scale", value=f"{tuple(np.ones(labels.ndim))}")
    
    for col in [c for c in props_table.columns if "intensity" in c]:
        props_table.rename(columns={col:col[:-1]+list_obj_names[int(col[-1])]+"-ch"}, inplace=True)
        
    # props_table.insert((props_table.columns.get_loc('object')+1), f'{mask_name}_number', value=cells)
    # props_table.insert((props_table.columns.get_loc(f'{mask_name}_number')+1), 'subregion', value=regions)
    
    ######################################################
    ## optional: DISTRIBUTION OF INTERACTION MEASUREMENTS
    ######################################################
    if include_dist:
        XY_distribution, XY_bins, XY_wedges = get_XY_distribution(mask=mask,
                                                                centering_obj=dist_centering_obj,
                                                                obj=labels,
                                                                obj_name=overlap_ID,
                                                                scale=scale,
                                                                num_bins=dist_num_bins,
                                                                center_on=dist_center_on,
                                                                keep_center_as_bin=dist_keep_center_as_bin,
                                                                zernike_degrees=dist_zernike_degrees)
        Z_distribution = get_Z_distribution(mask=mask, 
                                            obj=labels,
                                            obj_name=overlap_ID,
                                            center_obj=dist_centering_obj,
                                            scale=scale)
        interaction_dist_tab = pd.merge(XY_distribution, Z_distribution, on=["object", "scale"])

        indexes.clear()

        if return_site:
            return site, props_table, interaction_dist_tab, XY_bins, XY_wedges
        else:
            return props_table, interaction_dist_tab, XY_bins, XY_wedges
    else:
        indexes.clear()
        if return_site:
            return site, props_table
        else:
            return props_table
        

# from ZSC for implementation in future infer-subc v2.0.0
def all_combo_20250804(list_obj_names: list[str], splitter: str="X"):
    all_pos = []
    for n in list(map(lambda x:x+2, (range(len(list_obj_names)-1)))):
        all_pos += itertools.combinations(list_obj_names, n)
    possib = [splitter.join(inter) for inter in all_pos]
    return possib


# from ZSC for implementation in future infer-subc v2.0.0
def find_non_redundant_overlaps2_20250804(site: np.ndarray,
                                orgs: str,
                                organelle_segs: dict[str:np.ndarray],
                                splitter: str="X"):
    ##########################################
    ## DETERMINE REDUNDANT OVERLAPS
    ##########################################
    LOc_NR = site.copy()            
    for org, val in organelle_segs.items():         
        if (org not in orgs.split(splitter)
            and np.any(site.astype(int)*val.astype(int))):
            HOc = site.copy()       
            valid = (LOc_NR>0)*(val>0)                  
            HOc[valid.astype(bool)==False]=0
            for id in np.unique(HOc):
                LOc_NR[LOc_NR==id] = 0    
    return LOc_NR



# from ZSC for implementation in future infer-subc v2.0.0
def get_interaction_metrics_3D_20250804(list_obj_names: list[str],
                               list_obj_segs: list[np.ndarray],
                               intensity_img: np.ndarray,
                               channel_axis: int,
                               mask: np.ndarray,
                               mask_name: str,
                            #    regions: dict[str:np.ndarray],
                               splitter: str="X",
                               scale: Union[tuple, None]=None,
                               include_dist:bool=False, 
                               dist_centering_obj: Union[List[np.ndarray], None]=None,
                               dist_num_bins: Union[int, None]=None,
                               dist_zernike_degrees: Union[int, None]=None,
                               dist_center_on: Union[bool, None]=None,
                               dist_keep_center_as_bin: Union[bool, None]=None):

### UPDATE FOR 2.2 notebook single cell analysis ###############################
    ########################
    ## CREATE ORGANELLE DICT
    ########################
    organelle_segs = make_dict_20250804(list_obj_names, list_obj_segs)
    
    #########################
    ## LIST POSSIBLE OVERLAPS
    #########################
    possib = all_combo_20250804(list_obj_names, splitter)

    #######################
    ## ANALYZE ALL OVERLAPS
    #######################
    inter_tabs=[]
    dist_tabs=[]
    XY_bin = []
    XY_wedge = []
    if include_dist:
        for inter in possib:
            site, inter_tab, dist_tab, XY_bins, XY_wedges = interaction_metric_analysis_20250804(overlap_ID=inter,
                                                                                        list_obj_names=list_obj_names,
                                                                                        list_obj_segs=list_obj_segs,
                                                                                        intensity_img=intensity_img,
                                                                                        channel_axis=channel_axis,
                                                                                        mask=mask,
                                                                                        mask_name=mask_name,
                                                                                        # regions_dict=regions,
                                                                                        splitter=splitter,
                                                                                        scale=scale,
                                                                                        include_dist=True,
                                                                                        dist_centering_obj=dist_centering_obj,
                                                                                        dist_num_bins=dist_num_bins,
                                                                                        dist_zernike_degrees=dist_zernike_degrees,
                                                                                        dist_center_on=dist_center_on,
                                                                                        dist_keep_center_as_bin=dist_keep_center_as_bin,
                                                                                        return_site=True)
            ### UPDATE FOR 2.2 NOTEBOOK ###############################
            LOi_NR = find_non_redundant_overlaps2_20250804(site, inter, organelle_segs, splitter)
            LOi_NR = apply_mask((LOi_NR>0), mask).astype(int) * site                                                        # extracting the interaction site idx numbers that are nonredundant
            redundancy = inter_tab['idx'].isin(np.unique(LOi_NR[LOi_NR>0]).tolist())                                        # select only the positive integer values within the array
            inter_tab.insert((inter_tab.columns.get_loc('label')+1), "in_higher_order", list(map(bool, ~redundancy)))   # add the redundancy column, but the opposite: True = in the higher order
            inter_tab.drop(columns=['idx'], inplace=True)
            inter_tabs.append(inter_tab)
            dist_tabs.append(dist_tab)
            if len(XY_bin) == 0:
                XY_bin.append(XY_bins)
                XY_wedge.append(XY_wedges)

    else:
        for inter in possib:
            ### UPDATE FOR METHOD_INTERACTION NOTEBOOK ###############################
            site, inter_tab = interaction_metric_analysis_20250804(overlap_ID=inter,
                                                         list_obj_names=list_obj_names,
                                                         list_obj_segs=list_obj_segs,
                                                         intensity_img=intensity_img,
                                                         channel_axis=channel_axis,
                                                         mask=mask,
                                                         mask_name=mask_name,
                                                        #  regions_dict=regions,
                                                         splitter=splitter,
                                                         scale=scale,
                                                         include_dist=False,
                                                         return_site=True)
            LOi_NR = find_non_redundant_overlaps2_20250804(site, inter, organelle_segs, splitter)
            LOi_NR = apply_mask((LOi_NR>0), mask).astype(int) * site
            redundancy = inter_tab['idx'].isin(np.unique(LOi_NR[LOi_NR>0]).tolist())
            inter_tab.drop(columns=['idx'], inplace=True)
            inter_tab.insert((inter_tab.columns.get_loc('label')+1), "in_higher_order", list(map(bool, ~redundancy)))
            inter_tabs.append(inter_tab)

            # make sure they are empty if not including them
            dist_tabs = []
            XY_bin = []
            XY_wedge = []

    return inter_tabs, dist_tabs, XY_bin, XY_wedge



# this function was taken from SCohenLab/infer-subc v1.0.0 and editted to include:
# intensity measurements for each channel
def get_org_morphology_3D_20250804(segmentation_img: np.ndarray, 
                           seg_name: str, 
                           intensity_img, 
                           intensity_ch_name,
                           mask: np.ndarray, 
                           channel_axis: int,
                           scale: Union[tuple, None]=None):
    """
    Parameters
    ------------
    segmentation_img:
        a 3D (ZYX) np.ndarray of segmented objects 
    seg_name: str
        a name or nickname of the object being measured; this will be used for record keeping in the output table
    intensity_img:
        a 3D (ZYX) np.ndarray contain gray scale values from the "raw" image the segmentation is based on )single channel)
    mask:
        a 3D (ZYX) binary np.ndarray mask of the area to measure from
    scale: tuple, optional
        a tuple that contains the real world dimensions for each dimension in the image (Z, Y, X)


    Regionprops measurements:
    ------------------------
    ['label',
    'centroid',
    'bbox',
    'area',
    'equivalent_diameter',
    'extent',
    'feret_diameter_max',
    'euler_number',
    'convex_area',
    'solidity',
    'axis_major_length',
    'axis_minor_length',
    'max_intensity',
    'mean_intensity',
    'min_intensity']

    Additional measurements:
    -----------------------
    ['standard_deviation_intensity',
    'surface_area']


    Returns
    -------------
    pandas dataframe of containing regionprops measurements (columns) for each object in the segmentation image (rows) and the regionprops object
    
    """
    ###################################################
    ## MASK THE ORGANELLE OBJECTS THAT WILL BE MEASURED
    ###################################################
    # in case we sent a boolean mask (e.g. cyto, nucleus, cellmask)
    input_labels = _assert_uint16_labels(segmentation_img)

    # mask
    input_labels = apply_mask(input_labels, mask)

    ##########################################
    ## CREATE LIST OF REGIONPROPS MEASUREMENTS
    ##########################################
    # start with LABEL
    properties = ["label"]

    # add position
    properties = properties + ["centroid", "bbox"]

    # add area
    properties = properties + ["area", "equivalent_diameter"] # "num_pixels", 

    # add shape measurements
    properties = properties + ["extent", "euler_number", "solidity", "axis_major_length"] # ,"feret_diameter_max", "axis_minor_length"]

    # add intensity values (used for quality checks)
    properties = properties + ["min_intensity", "max_intensity", "mean_intensity"]

    #######################
    ## ADD EXTRA PROPERTIES
    #######################
    def standard_deviation_intensity(region, intensities):
        return np.std(intensities[region])

    extra_properties = [standard_deviation_intensity]

    ##################
    ## RUN REGIONPROPS
    ##################
    if channel_axis == len(scale):
        intensity_input = intensity_img
    else:
        intensity_input = np.moveaxis(intensity_img, channel_axis, -1)

    props = regionprops_table(input_labels, 
                           intensity_image=intensity_input, 
                           properties=properties,
                           extra_properties=extra_properties,
                           spacing=scale)

    props_table = pd.DataFrame(props)
    props_table.insert(0, "object", seg_name)
    props_table.rename(columns={"area": "volume"}, inplace=True)

    if scale is not None:
        round_scale = (round(scale[0], 4), round(scale[1], 4), round(scale[2], 4))
        props_table.insert(loc=2, column="scale", value=f"{round_scale}")
    else: 
        props_table.insert(loc=2, column="scale", value=f"{tuple(np.ones(segmentation_img.ndim))}") 

    for col in [c for c in props_table.columns if "intensity" in c]:
        props_table.rename(columns={col:col[:-1]+intensity_ch_name[int(col[-1])]+"-ch"}, inplace=True)

    ##################################################################
    ## RUN SURFACE AREA FUNCTION SEPARATELY AND APPEND THE PROPS_TABLE
    ##################################################################
    surface_area_tab = pd.DataFrame(surface_area_from_props(input_labels, props, scale))

    props_table.insert(12, "surface_area", surface_area_tab)
    props_table.insert(14, "SA_to_volume_ratio", props_table["surface_area"].div(props_table["volume"]))

    ################################################################
    ## ADD SKELETONIZATION OPTION FOR MEASURING LENGTH AND BRANCHING
    ################################################################


    return props_table




# this function was taken from SCohenLab/infer-subc v1.0.0 and editted to include:
# optional measurement types
# multi-way interactions analysis
### UPDATES FOR 2.2 notebook MULTI-cell analysis ###############################
def make_all_metrics_tables_20250804(source_file: str,
                                    list_obj_names: List[str],
                                    list_obj_segs: List[np.ndarray],
                                    list_intensity_img: List[np.ndarray],
                                    list_region_names: List[str],
                                    list_region_segs: List[np.ndarray],
                                    mask: str,
                                    channel_axis: int,
                                    scale: Union[tuple,None] = None,
                                    include_org_morph: bool = True,
                                    include_region_morph: bool = True,
                                    include_interactions: bool = True,
                                    include_distribution: bool = True,
                                    include_interact_dist: bool = True,
                                    dist_centering_obj:str = 'nuc', 
                                    dist_num_bins: int = 5,
                                    dist_center_on: bool = False,
                                    dist_keep_center_as_bin: bool = True,
                                    dist_zernike_degrees: Union[int, None] = None):
    """
    Measure the composition, morphology, distribution, and contacts of multiple organelles in a cell

    Parameters:
    ----------
    source_file: str
        file path; this is used for recorder keeping of the file name in the output data tables
    list_obj_names: List[str]
        a list of object names (strings) that will be measured; this should match the order in list_obj_segs
    list_obj_segs: List[np.ndarray]
        a list of 3D (ZYX) segmentation np.ndarrays that will be measured per cell; the order should match the list_obj_names 
    list_intensity_img: List[np.ndarray]
        a list of 3D (ZYX) grayscale np.ndarrays that will be used to measure fluoresence intensity in each region and object
    list_region_names: List[str]
        a list of region names (strings); these should include the mask (entire region being measured - usually the cell) 
        and other sub-mask regions from which we can meausure the objects in (ex - nucleus, neurites, soma, etc.). It should 
        also include the centering object used when created the XY distribution bins.
        The order should match the list_region_segs
    list_region_segs: List[np.ndarray]
        a list of 3D (ZYX) binary np.ndarrays of the region masks; the order should match the list_region_names.
    mask: str
        a str of which region name (contained in the list_region_names list) should be used as the main mask (e.g., cell mask)
    dist_centering_obj:str
        a str of which region name (contained in the list_region_names list) should be used as the centering object in 
        get_XY_distribution()
    dist_num_bins: int
        the number of concentric rings to draw between the centering object and edge of the mask in get_XY_distribution()
    dist_center_on: bool=False,
        for get_XY_distribution:
        True = distribute the bins from the center of the centering object
        False = distribute the bins from the edge of the centering object
    dist_keep_center_as_bin: bool=True
        for get_XY_distribution:
        True = include the centering object area when creating the bins
        False = do not include the centering object area when creating the bins
    dist_zernike_degrees: Union[int, None]=None
        for get_XY_distribution:
        the number of zernike degrees to include for the zernike shape descriptors; if None, the zernike measurements will not 
        be included in the output
    scale: Union[tuple,None] = None
        a tuple that contains the real world dimensions for each dimension in the image (Z, Y, X)
    include_contact_dist:bool=True
        whether to include the distribution of contact sites in get_contact_metrics_3d(); True = include contact distribution

    Returns:
    ----------
    4 Dataframes of measurements of organelle morphology, region morphology, contact morphology, and organelle/contact distributions

    """
    start = time.time()
    count = 0

    # segmentation image for all masking steps below
    mask_name = mask
    mask = list_region_segs[list_region_names.index(mask)]

    # create np.ndarray of intensity images
    raw_image = np.stack(list_intensity_img) ### <-- CONSIDER DOING THIS OUTSIDE OF THIS FUNCTION

    ######################
    # measure cell regions
    ######################
    if include_region_morph:
        # container for region data
        region_tabs = []
        for r, r_name in enumerate(list_region_names):
            region = list_region_segs[r]
            region_metrics = get_region_morphology_3D(region_seg=region, 
                                                    region_name=r_name,
                                                    channel_names=list_obj_names,
                                                    intensity_img=raw_image, 
                                                    mask=mask,
                                                    scale=scale)
            region_tabs.append(region_metrics)

    ##############################################################
    # loop through all organelles to collect measurements for each
    ##############################################################
    # containers to collect per organelle information
    org_tabs = []
    dist_tabs = []
    inter_tabs = []
    XY_bins = []
    XY_wedges = []

    ### UPDATE FOR METHOD_MORPHOLOGY FUNCTION ###############################
    for j, target in enumerate(list_obj_names):
        # organelle intensity image
        # org_img = list_intensity_img[j]

        # organelle segmentation
        if target == 'ER':
            # ensure ER is only one object
            org_obj = (list_obj_segs[j] > 0).astype(np.uint16)
        else:
            org_obj = list_obj_segs[j]

        ##########################################################
        # measure organelle morphology & number of objs contacting
        ##########################################################
        if include_org_morph:
            org_metrics = get_org_morphology_3D_20250804(segmentation_img=org_obj, seg_name=target,
                                                         intensity_img=raw_image,
                                                         channel_axis=channel_axis,
                                                         intensity_ch_name=list_obj_names,
                                                         mask=mask,
                                                         scale=scale)

            ### org_metrics.insert(loc=0,column='cell',value=1) 
            # ^^^ saving this thought for later when someone might have more than one cell per image.
            # Not sure how they analysis process would fit in our pipelines as they exist now. 
            # Maybe here, iterating though the index of the masks above all of this and using that index as the cell number?

            org_tabs.append(org_metrics)

        ################################
        # measure organelle distribution 
        ################################
        if include_distribution:
            centering = list_region_segs[list_region_names.index(dist_centering_obj)]
            XY_org_distribution, XY_bin_masks, XY_wedge_masks = get_XY_distribution(mask=mask,
                                                                                    centering_obj=centering,
                                                                                    obj=org_obj,
                                                                                    obj_name=target,
                                                                                    scale=scale,
                                                                                    num_bins=dist_num_bins,
                                                                                    center_on=dist_center_on,
                                                                                    keep_center_as_bin=dist_keep_center_as_bin,
                                                                                    zernike_degrees=dist_zernike_degrees)
            Z_org_distribution = get_Z_distribution(mask=mask, 
                                                    obj=org_obj,
                                                    obj_name=target,
                                                    center_obj=centering,
                                                    scale=scale)
            
            org_distribution_metrics = pd.merge(XY_org_distribution, Z_org_distribution,on=["object", "scale"])

            dist_tabs.append(org_distribution_metrics)
            if len(XY_bins) == 0:
                XY_bins.append(XY_bin_masks)
            if len(XY_wedges) == 0:
                XY_wedges.append(XY_wedge_masks)
            # XY_bins = XY_bins | XY_bin_masks
            # XY_wedges = XY_wedges | XY_wedge_masks

    #######################################
    # collect non-redundant contact metrics 
    #######################################
    if include_interactions and len(list_obj_names)>2:
        if include_interact_dist:
            centering = list_region_segs[list_region_names.index(dist_centering_obj)]
            interaction_tabs, interaction_dist_tabs, dist_XY_bins, dist_XY_wedges = get_interaction_metrics_3D_20250804(list_obj_names=list_obj_names,
                                                                                                                list_obj_segs=list_obj_segs, 
                                                                                                                intensity_img=raw_image,
                                                                                                                channel_axis=channel_axis,
                                                                                                                mask=mask,
                                                                                                                mask_name=mask_name,
                                                                                                                # regions=locations,
                                                                                                                scale=scale,
                                                                                                                include_dist=include_interact_dist, 
                                                                                                                dist_centering_obj=centering,
                                                                                                                dist_num_bins=dist_num_bins,
                                                                                                                dist_zernike_degrees=dist_zernike_degrees,
                                                                                                                dist_center_on=dist_center_on,
                                                                                                                dist_keep_center_as_bin=dist_keep_center_as_bin)
            for tab in interaction_dist_tabs:
                dist_tabs.append(tab)
            for tab in interaction_tabs:
                inter_tabs.append(tab)
            if len(XY_bins) == 0:      
                XY_bins = dist_XY_bins
            if len(XY_wedges) == 0:
                XY_wedges = dist_XY_wedges

        else:
            interaction_tabs = get_interaction_metrics_3D_20250804(list_obj_names=list_obj_names,
                                                            list_obj_segs=list_obj_segs,
                                                            intensity_img=raw_image,
                                                            channel_axis=channel_axis,
                                                            mask=mask,
                                                            mask_name=mask_name,
                                                            # regions=locations,
                                                            scale=scale,
                                                            include_dist=False, 
                                                            dist_centering_obj=None,
                                                            dist_num_bins=None,
                                                            dist_zernike_degrees=None,
                                                            dist_center_on=None,
                                                            dist_keep_center_as_bin=None)
            for tab in interaction_tabs[0]:
                inter_tabs.append(tab)


    ###########################################
    # combine all tabs into one table per type:
    ###########################################
    if include_org_morph:
        final_org_tab = pd.concat(org_tabs, ignore_index=True)
        final_org_tab.insert(loc=0,column='image_name',value=source_file.stem)
    else:
        final_org_tab = []

    if include_interactions:
        final_interaction_tab = pd.concat(inter_tabs, ignore_index=True)
        final_interaction_tab.insert(loc=0,column='image_name',value=source_file.stem)
    else: 
        final_interaction_tab = []

    if include_distribution or include_interact_dist:
        combined_dist_tab = pd.concat(dist_tabs, ignore_index=True)
        combined_dist_tab.insert(loc=0,column='image_name',value=source_file.stem)
    else:
        combined_dist_tab = []
        XY_bins = []
        XY_wedges = []

    if include_region_morph:
        final_region_tab = pd.concat(region_tabs, ignore_index=True)
        final_region_tab.insert(loc=0,column='image_name',value=source_file.stem)
    else:
        final_region_tab = []

    end = time.time()
    print(f"It took {(end-start)/60} minutes to quantify one image.")
    return final_org_tab, final_interaction_tab, combined_dist_tab, final_region_tab, XY_bins, XY_wedges




# this function was taken from SCohenLab/infer-subc v1.0.0 and editted 
def batch_process_quantification_20250804(out_file_name: str,
                                        seg_path: Union[Path,str],
                                        out_path: Union[Path, str], 
                                        raw_path: Union[Path,str], 
                                        raw_file_type: str,
                                        organelle_names: List[str],
                                        organelle_channels: List[int],
                                        region_names: List[str],
                                        masks_file_name: List[str],
                                        mask: str,
                                        scale:bool=True,
                                        seg_suffix:Union[str, None]=None,
                                        include_org_morph: bool = True,
                                        include_region_morph: bool = True,
                                        include_interactions: bool = True,
                                        include_distribution: bool = True,
                                        include_interact_dist: bool = True,
                                        dist_centering_obj:str = 'nuc', 
                                        dist_num_bins: int = 5,
                                        dist_center_on: bool = False,
                                        dist_keep_center_as_bin: bool = True,
                                        dist_zernike_degrees: Union[int, None] = None) -> int :
    """  
    batch process segmentation quantification (morphology, distribution, contacts); this function is currently optimized to process images from one file folder per image type (e.g., raw, segmentation)
    the output csv files are saved to the indicated out_path folder

    Parameters:
    ----------
    out_file_name: str
        the prefix to use when naming the output datatables
    seg_path: Union[Path,str]
        Path or str to the folder that contains the segmentation tiff files
    out_path: Union[Path, str]
        Path or str to the folder that the output datatables will be saved to
    raw_path: Union[Path,str]
        Path or str to the folder that contains the raw image files
    raw_file_type: str
        the file type of the raw data; ex - ".tiff", ".czi"
    organelle_names: List[str]
        a list of all organelle names that will be analyzed; the names should be the same as the suffix used to name each of the tiff segmentation files
        Note: the intensity measurements collect per region (from get_region_morphology_3D function) will only be from channels associated to these organelles 
    organelle_channels: List[int]
        a list of channel indices associated to respective organelle staining in the raw image; the indices should listed in same order in which the respective segmentation name is listed in organelle_names
    region_names: List[str]
        a list of regions, or masks, to measure; the order should correlate to the order of the channels in the "masks" output segmentation file
    masks_file_name: str
        the suffix of the "masks" segmentation file; ex- "masks_B", "masks", etc.
        this function currently does not accept indivial region segmentations 
    mask: str
        the name of the region to use as the mask when measuring the organelles; this should be one of the names listed in regions list; usually this will be the "cell" mask
    dist_centering_obj:str
        the name of the region or object to use as the centering object in the get_XY_distribution function
    dist_num_bins: int
        the number of bins for the get_XY_distribution function
    dist_center_on: bool=False,
        for get_XY_distribution:
        True = distribute the bins from the center of the centering object
        False = distribute the bins from the edge of the centering object
    dist_keep_center_as_bin: bool=True
        for get_XY_distribution:
        True = include the centering object area when creating the bins
        False = do not include the centering object area when creating the bins
    dist_zernike_degrees: Union[int, None]=None
        for get_XY_distribution:
        the number of zernike degrees to include for the zernike shape descriptors; if None, the zernike measurements will not 
        be included in the output
    include_contact_dist:bool=True
        whether to include the distribution of contact sites in get_contact_metrics_3d(); True = include contact distribution
    scale:bool=True
        a tuple that contains the real world dimensions for each dimension in the image (Z, Y, X)
    seg_suffix:Union[str, None]=None
        any additional text that is included in the segmentation tiff files between the file stem and the segmentation suffix
    


    Returns:
    ----------
    count: int
        the number of images processed
        
    """
    ### BATCH FUNCTION ###############################
    start = time.time()
    count = 0

    if isinstance(raw_path, str): raw_path = Path(raw_path)
    if isinstance(seg_path, str): seg_path = Path(seg_path)
    if isinstance(out_path, str): out_path = Path(out_path)
    
    if not Path.exists(out_path):
        Path.mkdir(out_path)
        print(f"making {out_path}")
    
    # reading list of files from the raw path
    img_file_list = list_image_files(raw_path, raw_file_type)

    # list of segmentation files to collect
    segs_to_collect = organelle_names + masks_file_name
    
    # containers to collect data tabels
    org_tabs = []
    contact_tabs = []
    dist_tabs = []
    region_tabs = []

    for img_f in img_file_list:
        count = count + 1
        filez = find_segmentation_tiff_files(img_f, segs_to_collect, seg_path, seg_suffix)

        # read in raw file and metadata
        img_data, meta_dict = read_czi_image(filez["raw"])
        channel_axis = meta_dict['channel_axis']

        # create intensities from raw file as list based on the channel order provided
        intensities = [img_data[ch] for ch in organelle_channels]

        # define the scale
        if scale is True:
            scale_tup = meta_dict['scale']
        else:
            scale_tup = None

        # load regions as a list based on order in list (should match order in "masks" file)
        masks = [] 
        for m in masks_file_name:
            mfile = read_tiff_image(filez[m])
            masks.append(mfile)
        regions = [masks[r] for r, region in enumerate(region_names)]

        # store organelle images as list
        organelles = [read_tiff_image(filez[org]) for org in organelle_names]

        org_metrics, contact_metrics, dist_metrics, region_metrics, XY_bins, XY_wedges = make_all_metrics_tables_20250804(source_file=img_f,
                                                                                                                        list_obj_names=organelle_names,
                                                                                                                        list_obj_segs=organelles,
                                                                                                                        list_intensity_img=intensities, 
                                                                                                                        list_region_names=region_names,
                                                                                                                        list_region_segs=regions, 
                                                                                                                        channel_axis=channel_axis,
                                                                                                                        mask=mask,
                                                                                                                        scale=scale_tup,
                                                                                                                        include_org_morph=include_org_morph,
                                                                                                                        include_region_morph=include_region_morph,
                                                                                                                        include_interactions=include_interactions,
                                                                                                                        include_distribution=include_distribution,
                                                                                                                        include_interact_dist=include_interact_dist,
                                                                                                                        dist_centering_obj=dist_centering_obj,
                                                                                                                        dist_num_bins=dist_num_bins,
                                                                                                                        dist_center_on=dist_center_on,
                                                                                                                        dist_keep_center_as_bin=dist_keep_center_as_bin,
                                                                                                                        dist_zernike_degrees=dist_zernike_degrees)

        org_tabs.append(org_metrics)
        contact_tabs.append(contact_metrics)
        dist_tabs.append(dist_metrics)
        region_tabs.append(region_metrics)

        dist_mask_path = out_path / "XY_distribution_bins_wedges"
        if not Path.exists(dist_mask_path):
            Path.mkdir(dist_mask_path)
            print(f"making {dist_mask_path}")
        if XY_bins is not None:
            out_bins = export_inferred_organelle(XY_bins[0], "XY_bins", meta_dict, dist_mask_path)
        if XY_wedges is not None:
            out_bins = export_inferred_organelle(XY_wedges[0], "XY_wedges", meta_dict, dist_mask_path)

        end2 = time.time()
        print(f"Completed processing for {count} images in {(end2-start)/60} mins.")

    if include_org_morph:
        final_org = pd.concat(org_tabs, ignore_index=True)
        org_csv_path = out_path / f"{out_file_name}_organelles.csv"
        final_org.to_csv(org_csv_path)
    
    if include_interactions:
        final_contact = pd.concat(contact_tabs, ignore_index=True)
        contact_csv_path = out_path / f"{out_file_name}_interactions.csv"
        final_contact.to_csv(contact_csv_path)
    
    if include_distribution or include_interact_dist:
        final_dist = pd.concat(dist_tabs, ignore_index=True)
        dist_csv_path = out_path / f"{out_file_name}_distributions.csv"
        final_dist.to_csv(dist_csv_path)

    if include_region_morph:
        final_region = pd.concat(region_tabs, ignore_index=True)
        region_csv_path = out_path / f"{out_file_name}_regions.csv"
        final_region.to_csv(region_csv_path)

    end = time.time()
    print(f"Quantification for {count} files is COMPLETE! Files saved to '{out_path}'.")
    print(f"It took {(end - start)/60} minutes to quantify these files.")
    return count #, XY_bins, XY_wedges, final_org, final_contact, final_dist, final_region




# NEW NEW: quick QC function
def QC_mask_quick(out_file_name: str,
                seg_path: Union[Path,str],
                out_path: Union[Path, str], 
                raw_path: Union[Path,str], 
                raw_file_type: str,
                organelle_names: List[str],
                organelle_channels: List[int],
                region_names: List[str],
                masks_file_name: List[str],
                mask: str,
                scale:bool=True,
                seg_suffix:Union[str, None]=None,
                include_org_morph: bool = True,
                include_region_morph: bool = True,
                include_interactions: bool = True,
                include_distribution: bool = True,
                include_interact_dist: bool = True,
                dist_centering_obj:str = 'nuc', 
                dist_num_bins: int = 5,
                dist_center_on: bool = False,
                dist_keep_center_as_bin: bool = True,
                dist_zernike_degrees: Union[int, None] = None):
    
    count = 0
    if isinstance(raw_path, str): raw_path = Path(raw_path)
    if isinstance(seg_path, str): seg_path = Path(seg_path)
    if isinstance(out_path, str): out_path = Path(out_path)
    
    if not Path.exists(out_path):
        Path.mkdir(out_path)
        print(f"making {out_path}")
    
    # reading list of files from the raw path
    img_file_list = list_image_files(raw_path, raw_file_type)

    # list of segmentation files to collect
    # segs_to_collect = organelle_names + masks_file_name

    for img_f in img_file_list:
        print(f"Beginning check for: {img_f}")
        count = count + 1
        # filez = find_segmentation_tiff_files(img_f, segs_to_collect, seg_path, seg_suffix)
        filez = find_segmentation_tiff_files(img_f, masks_file_name, seg_path, seg_suffix)

        # load regions as a list based on order in list (should match order in "masks" file)
        masks = [] 
        for m in masks_file_name:
            mfile = read_tiff_image(filez[m])
            masks.append(mfile)
        regions = [masks[r] for r, region in enumerate(region_names)]

        # segmentation image for all masking steps below
        mask_name = mask
        mask_obj = regions[region_names.index(mask)]

        # CHECKING ONLY ONE MASK OBJECT PER IMAGE
        mask_obj_unique = np.unique(mask_obj).tolist()
        mask_obj_unique.remove(0)

        if len(mask_obj_unique) > 1:
            print(f"ERROR: {mask_name} has more than on object in image: {img_f}")

        # CHECKING ONLY ONE CENTERING OBJECT PER MASK
        centering_obj = regions[region_names.index(dist_centering_obj)]
        centering_obj = apply_mask(centering_obj, mask_obj)

        centering_obj_unique = np.unique(mask_obj).tolist()
        centering_obj_unique.remove(0)

        if len(centering_obj_unique) > 1:
            print(f"ERROR: distribution centering object {dist_centering_obj} has MORE THAN ONE object per mask in image: {img_f}")
        elif len(centering_obj_unique) == 0:
            print(f"ERROR: distribution centering object {dist_centering_obj} is MISSING from the mask in image: {img_f}")
    
    return count

        

In [ ]:
#### FUNCTIONS FOR BATCH_SUMMARY_STATS THAT NEEDED EDITING ####
###############################################################

# originally created by ZSC for infer-subc v2.0.0 implementation
def batch_summary_interactions_2_20250805(interaction_df, splitter, org_list):
    
    # CREATE A NEW DATA TABLE FOR INTERACTION COUNTS
    interaction_cnt = interaction_df[["dataset", "image_name", "object", "label", "volume"]]
    
    # CREATES NEW COLUMNS EQUAL TO MAX NUMBER OF ORGANELLES INVOLVED IN A CONTACT
    # CREATES A NEW COLUMN FOR STORING THE ORGANELLE ID FOR EACH ORGANELLE INVOLVED IN A CONTACT
    # FOR EXAMPLE: mitoXER of 06_01 would become the following
    #              A_ID | orgA | B_ID | orgB
    #               06  | mito |  01  |  ER 
    interaction_cnt[[f"org{cha}" for cha in string.ascii_uppercase[:(len(max(interaction_cnt["object"].str.split(splitter), key=len)))]]] = interaction_cnt["object"].str.split(splitter, expand=True)
    interaction_cnt[[f"{cha}_ID" for cha in string.ascii_uppercase[:(len(max(interaction_cnt["label"].str.split('_'), key=len)))]]] = interaction_cnt["label"].str.split('_', expand=True)
    #iterating from a to val
    unstacked_interactions = []
    for cha in string.ascii_uppercase[:len(max(interaction_cnt["object"].str.split(splitter), key=len))]:
        # DETERMINE WHICH VALUES IN interaction_cnt HAVE THE DESIRED ALPHABETICAL ORGANELLE COUNT
        # i.e. if a contact has 4 organelles in it, the maximum alphabetical organelle count will be "D"
        valid = (interaction_cnt[f"org{cha}"] != None) & (interaction_cnt[f"{cha}_ID"] != None)

        # CREATES A NEW COLUMN WITH THE VALUE OF THE CURRENT ALPHABETICAL CHARACTER
        interaction_cnt[f"{cha}"] = None

        # Note: USED LATER WITH SEPARATING ORG AND INTERACTION DATA
        interaction_cnt.loc[valid, f"{cha}"] = interaction_cnt[f"org{cha}"] + "_" + interaction_cnt[f"{cha}_ID"]

        # CREATES A NEW DATAFRAME FOR PER CELL DATA FOCUSING ONLY ON CURRENT ALPHABETICAL ORGANELLE CHARACTER GROUPED BY CELL
        interaction_cnt_percell = interaction_cnt[["dataset", "image_name", f"org{cha}", f"{cha}_ID", "object", "volume"]].groupby(["dataset", "image_name", f"org{cha}", f"{cha}_ID", "object"]).agg(["count", "sum"])
        interaction_cnt_percell.columns = ["_".join(col_name).rstrip('_') for col_name in interaction_cnt_percell.columns.to_flat_index()]
        unstacked = interaction_cnt_percell.unstack(level='object')
        unstacked.columns = ["_".join(col_name).rstrip('_') for col_name in unstacked.columns.to_flat_index()]
        unstacked = unstacked.reset_index()
        for col in unstacked.columns:
            if col.startswith("volume_count_"):
                newname = col.split("_")[-1] + "_count"
                unstacked.rename(columns={col:newname}, inplace=True)
            if col.startswith("volume_sum_"):
                newname = col.split("_")[-1] + "_volume"
                unstacked.rename(columns={col:newname}, inplace=True)
        unstacked.rename(columns={f"org{cha}":"object", f"{cha}_ID":"label"}, inplace=True)
        unstacked.set_index(['dataset', 'image_name', 'object', 'label'])    
        unstacked_interactions.append(unstacked)
    interaction_cnt = pd.concat(unstacked_interactions, axis=0).sort_index(axis=0)
    interaction_cnt = interaction_cnt.groupby(['dataset', 'image_name', 'object', 'label']).sum().reset_index()                 #adds together all duplicates at the index, then resets the index
    interaction_cnt['label'] = interaction_cnt['label'].astype("Int64")

    # ensure all possible combinations of organelle interactions are present
    all_pos = all_combo_20250804(org_list, splitter=splitter)
    for col in all_pos:
        if col not in [col.split("_")[0] for col in interaction_cnt.columns if col.endswith('_count')]:
            interaction_cnt[col + "_count"] = 0
        if col not in [col.split("_")[0] for col in interaction_cnt.columns if col.endswith('_volume')]:
            interaction_cnt[col + "_volume"] = 0

    return interaction_cnt



# originally created by ZSC for infer-subc v2.0.0 implementation
def normalize_interaction_volumes_20250805(interaction_summary, 
                                           org_summary,
                                            # region_summary, 
                                            # regions_df,
                                            # group_by,
                                            splitter: str="X"):
    # regions_df = regions_df.set_index(group_by + ['label'])
    # for region in interaction_summary.columns.get_level_values(0).unique():
    norm_to_list = {}
    for idx,cha in enumerate(string.ascii_uppercase[:len(max(interaction_summary.index.get_level_values('object').str.split(splitter), key=len))]):
        for row in interaction_summary.index:
            if cha not in norm_to_list:
                norm_to_list[cha]=[]
            if ((idx+1) <= len(row[-1].split(splitter))): # continue if nth order 
                org = row[-1].split(splitter)[idx]
                if (interaction_summary.loc[row][('volume', 'sum')]>=0): # and (sum([region_summary.loc[row[:-1]+(reg.split('-')[0],)][(f"{org}_volume", 'sum')] for reg in region.split(':')[-1].split("_")]) >= 0):
                    norm_to_list[cha].append(interaction_summary.loc[row][('volume', 'sum')]/org_summary.loc[row[:-1] + (org,)][("volume", "sum")] ) #sum([region_summary.loc[row[:-1]+(reg.split('-')[0],)][(f"{org}_volume", 'sum')] for reg in region.split(':')[-1].split("_")]))
                else:
                    norm_to_list[cha].append(None)
            else: # specified interaction is below nth order, leave cell as none
                norm_to_list[cha].append(None)
    for cha in string.ascii_uppercase[:len(max(interaction_summary.index.get_level_values('object').str.split(splitter), key=len))]:
        interaction_summary[('volume', f'norm_to_{cha}')] = norm_to_list[cha]
    interaction_summary = interaction_summary.reindex(interaction_summary.columns.get_level_values(0).unique(), level=0, axis=1)
    return interaction_summary



# originally from SCohenLab/infer-subc v1.0.0 with updates to:
# normalize multi-way interaction site volumes
# calculate multi-way interaction site volume fractions
### UPDATES FOR 2.2 NOTEBOOK SUMMARY STATISTICS ########################################
def batch_summary_stats_20250805(csv_path_list: List[str],
                                out_path: str,
                                out_preffix: str,
                                mask_name: str):
    """" 
    csv_path_list: List[str],
        A list of path strings where .csv files to analyze are located.
    out_path: str,
        A path string where the summary data file will be output to
    out_preffix: str
        The prefix used to name the output file.    
    """
    ds_count = 0
    fl_count = 0
    ###################
    # Read in the csv files and combine them into one of each type
    ###################
    org_tabs = []
    contact_tabs = []
    dist_tabs = []
    region_tabs = []

    org = "_organelles"
    contacts = "_interactions"
    dist = "_distributions"
    regions = "_regions"

    for loc in csv_path_list:
        ds_count = ds_count + 1
        loc=Path(loc)
        files_store = sorted(loc.glob("*.csv"))
        for file in files_store:
            fl_count = fl_count + 1
            stem = file.stem

            if org in stem:
                test_orgs = pd.read_csv(file, index_col=0)
                test_orgs.insert(0, "dataset", stem[:-11])
                org_tabs.append(test_orgs)
            if contacts in stem:
                test_contact = pd.read_csv(file, index_col=0)
                test_contact.insert(0, "dataset", stem[:-13])
                contact_tabs.append(test_contact)
            if dist in stem:
                test_dist = pd.read_csv(file, index_col=0)
                test_dist.insert(0, "dataset", stem[:-14])
                dist_tabs.append(test_dist)
            if regions in stem:
                test_regions = pd.read_csv(file, index_col=0)
                test_regions.insert(0, "dataset", stem[:-8])
                region_tabs.append(test_regions)
            
    org_df = pd.concat(org_tabs,axis=0, join='outer')
    contacts_df = pd.concat(contact_tabs,axis=0, join='outer')
    dist_df = pd.concat(dist_tabs,axis=0, join='outer')
    regions_df = pd.concat(region_tabs,axis=0, join='outer')

    ###################
    # adding new metrics to the original sheets
    ###################    
    org_list = org_df['object'].unique().tolist()
    contact_cnt = batch_summary_interactions_2_20250805(contacts_df, 'X', org_list)

    org_df = pd.merge(org_df, contact_cnt, on=['dataset', 'image_name', 'object', 'label'], how='left',sort=True)
    org_df[contact_cnt.columns] = org_df[contact_cnt.columns].fillna(0)


    ###################
    # summary stat group
    ###################
    group_by = ['dataset', 'image_name', 'object']
    sharedcolumns = ["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"]
    ag_func_standard = ['mean', 'median', 'std']

    ###################
    # summarize shared measurements between org_df and contacts_df
    ###################
    org_cont_tabs = []
    for tab in [org_df, contacts_df]:
        tab1 = tab[group_by + ['volume']].groupby(group_by).agg(['count', 'sum'] + ag_func_standard)
        tab2 = tab[group_by + ['surface_area']].groupby(group_by).agg(['sum'] + ag_func_standard)
        tab3 = tab[group_by + sharedcolumns].groupby(group_by).agg(ag_func_standard)
        shared_metrics = pd.merge(tab1, tab2, 'outer', on=group_by)
        shared_metrics = pd.merge(shared_metrics, tab3, 'outer', on=group_by)
        org_cont_tabs.append(shared_metrics)

    org_summary = org_cont_tabs[0]
    contact_summary = org_cont_tabs[1]

    ###################
    # group metrics from regions_df similar to the above
    ###################
    regions_summary = regions_df[group_by + ['volume', 'surface_area'] + sharedcolumns].set_index(group_by)
    regions_summary.sort_index(axis=1, inplace=True)

    ###################
    # summarize extra metrics from org_df
    ###################
    columns2 = [col for col in org_df.columns if col.endswith(("_count", "_volume"))]
    contact_counts_summary = org_df[group_by + columns2].groupby(group_by).agg(['sum'] + ag_func_standard)
    org_summary = pd.merge(org_summary, contact_counts_summary, 'outer', on=group_by)


    ###################
    # summarize distribution measurements
    ###################
    # organelle distributions
    hist_dfs = []
    for ind in range(0,len(dist_df.index)):
        selection = dist_df.iloc[[ind]] #    selection = dist_df.loc[[ind]]
        bins_df = pd.DataFrame()
        wedges_df = pd.DataFrame()
        Z_df = pd.DataFrame()
        CV_df = pd.DataFrame()

        bins_df[['bins', 'masks', 'obj']] = selection[['XY_bins', 'XY_mask_vox_cnt_perbin', 'XY_obj_vox_cnt_perbin']]
        wedges_df[['bins', 'masks', 'obj']] = selection[['XY_wedges', 'XY_mask_vox_cnt_perwedge', 'XY_obj_vox_cnt_perwedge']]
        Z_df[['bins', 'masks', 'obj']] = selection[['Z_slices', 'Z_mask_vox_cnt', 'Z_obj_vox_cnt']]

        dfs = [selection[['dataset', 'image_name', 'object']].reset_index()]
        for df, prefix in zip([bins_df, wedges_df, Z_df], ["XY_bins_", "XY_wedges_", "Z_slices_"]):
            single_df = pd.DataFrame(list(zip(df["bins"].values[0][1:-1].split(", "), 
                                            df["obj"].values[0][1:-1].split(", "), 
                                            df["masks"].values[0][1:-1].split(", "))), columns =['bins', 'obj', 'mask']).astype(int)
            
            if "Z_" in prefix:
                single_df =  single_df.drop(single_df[single_df['mask'] == 0].index)
                single_df['bins'] = (single_df["bins"]/max(single_df.bins)*9.99).apply(np.floor)+1
                single_df = single_df.groupby("bins").agg(['sum']).reset_index()
                single_df.columns = ['bins',"obj","mask"]
        
            single_df['mask_fract'] = single_df['mask']/single_df['mask'].max()
            # single_df['obj_normed_tocell'] = (single_df["obj"]*single_df["mask_fract"]).fillna(0)
            single_df['obj_perc_per_bin'] = (single_df["obj"] / single_df["obj"].sum())*100
            single_df['obj_portion_normed_tobin'] = (single_df["obj_perc_per_bin"]/single_df["mask_fract"]).fillna(0)

            sumstats_df = pd.DataFrame()

            s = single_df['bins'].repeat(single_df['obj_portion_normed_tobin']*100)

            sumstats_df['hist_mean']=[s.mean()]
            sumstats_df['hist_median']=[s.median()]
            if single_df['obj_portion_normed_tobin'].sum() != 0: sumstats_df['hist_mode']=[s.mode().iloc[0]]
            else: sumstats_df['hist_mode']=['NaN']
            sumstats_df['hist_min']=[s.min()]
            sumstats_df['hist_max']=[s.max()]
            sumstats_df['hist_range']=[s.max() - s.min()]
            sumstats_df['hist_stdev']=[s.std()]
            sumstats_df['hist_skew']=[s.skew()]
            sumstats_df['hist_kurtosis']=[s.kurtosis()]
            sumstats_df['hist_var']=[s.var()]
            sumstats_df.columns = [prefix+col for col in sumstats_df.columns]
            dfs.append(sumstats_df.reset_index())

        CV_df = pd.DataFrame(list(zip(selection["XY_obj_cv_perbin"].values[0][1:-1].split(", "))), columns =['CV']).astype(float)
        sumstats_CV_df = pd.DataFrame()
        sumstats_CV_df['XY_bin_CV_mean'] = CV_df.mean()
        sumstats_CV_df['XY_bin_CV_median'] = CV_df.median()
        sumstats_CV_df['XY_bin_CV_std'] = CV_df.std()
        dfs.append(sumstats_CV_df.reset_index().drop(['index'], axis=1))

        combined_df = pd.concat(dfs, axis=1).drop(columns="index")
        hist_dfs.append(combined_df)
    dist_org_summary = pd.concat(hist_dfs, ignore_index=True)
    dist_org_summary

    # nucleus distribution
    nuc_dist_df = dist_df[["dataset", "image_name", 
                        "XY_bins", "XY_center_vox_cnt_perbin", "XY_mask_vox_cnt_perbin",
                        "XY_wedges", "XY_center_vox_cnt_perwedge", "XY_mask_vox_cnt_perwedge",
                        "Z_slices", "Z_center_vox_cnt", "Z_mask_vox_cnt"]].set_index(["dataset", "image_name"])
    nuc_hist_dfs = []
    for idx in nuc_dist_df.index.unique():
        selection = nuc_dist_df.loc[idx].iloc[[0]].reset_index()
        bins_df = pd.DataFrame()
        wedges_df = pd.DataFrame()
        Z_df = pd.DataFrame()

        bins_df[['bins', 'center', 'masks']] = selection[['XY_bins', 'XY_center_vox_cnt_perbin', 'XY_mask_vox_cnt_perbin']]
        wedges_df[['bins', 'center', 'masks']] = selection[['XY_wedges', 'XY_center_vox_cnt_perwedge', 'XY_mask_vox_cnt_perwedge']]
        Z_df[['bins', 'center', 'masks']] = selection[['Z_slices', 'Z_center_vox_cnt', 'Z_mask_vox_cnt']]

        dfs = [selection[['dataset', 'image_name']]]
        for df, prefix in zip([bins_df, wedges_df, Z_df], ["XY_bins_", "XY_wedges_", "Z_slices_"]):
            single_df = pd.DataFrame(list(zip(df["bins"].values[0][1:-1].split(", "), 
                                            df["masks"].values[0][1:-1].split(", "),
                                            df["center"].values[0][1:-1].split(", "))), columns =['bins', 'mask', 'obj']).astype(int)

            if "Z_" in prefix:
                single_df =  single_df.drop(single_df[single_df['mask'] == 0].index)
                single_df['bins'] = (single_df["bins"]/max(single_df.bins)*9.99).apply(np.floor)+1
                single_df = single_df.groupby("bins").agg(['sum']).reset_index()
                single_df.columns = ['bins',"mask","obj"]
        
            single_df['mask_fract'] = single_df['mask']/single_df['mask'].max()
            # single_df['obj_normed_tocell'] = (single_df["obj"]*single_df["mask_fract"]).fillna(0)
            single_df['obj_perc_per_bin'] = (single_df["obj"] / single_df["obj"].sum())*100
            single_df['obj_portion_normed_tobin'] = (single_df["obj_perc_per_bin"]/single_df["mask_fract"]).fillna(0)

            sumstats_df = pd.DataFrame()

            s = single_df['bins'].repeat(single_df['obj_portion_normed_tobin']*100)

            sumstats_df['hist_mean']=[s.mean()]
            sumstats_df['hist_median']=[s.median()]
            if single_df['obj_portion_normed_tobin'].sum() != 0: sumstats_df['hist_mode']=[s.mode().iloc[0]]
            else: sumstats_df['hist_mode']=['NaN']
            sumstats_df['hist_min']=[s.min()]
            sumstats_df['hist_max']=[s.max()]
            sumstats_df['hist_range']=[s.max() - s.min()]
            sumstats_df['hist_stdev']=[s.std()]
            sumstats_df['hist_skew']=[s.skew()]
            sumstats_df['hist_kurtosis']=[s.kurtosis()]
            sumstats_df['hist_var']=[s.var()]
            sumstats_df.columns = [prefix+col for col in sumstats_df.columns]
            dfs.append(sumstats_df.reset_index())
        combined_df = pd.concat(dfs, axis=1).drop(columns="index")
        nuc_hist_dfs.append(combined_df)
    dist_center_summary = pd.concat(nuc_hist_dfs, ignore_index=True)
    dist_center_summary.insert(2, column="object", value="nuc")

    dist_summary = pd.concat([dist_org_summary, dist_center_summary], axis=0).set_index(group_by).sort_index()
    dist_summary.sort_index(axis=1, inplace=True)


    ###################
    # add normalization
    ###################
    # organelle area fraction
    area_fractions = []
    for idx in org_summary.index.unique():
        org_vol = org_summary.loc[idx][('volume', 'sum')]
        cell_vol = regions_summary.loc[idx[:-1] + (mask_name,)]["volume"]
        afrac = org_vol/cell_vol
        area_fractions.append(afrac)
    org_summary[('volume', 'fraction')] = area_fractions
    org_summary.sort_index(axis=1, inplace=True)


    # number and area of individuals organelle involved in contact
    contact_summary = normalize_interaction_volumes_20250805(contact_summary, org_summary)

    all_pos = all_combo_20250804(org_list, "X")
    for ind in contact_summary.index.droplevel(2).unique().to_list():
        for row in all_pos:
            if ind+(row,) not in contact_summary.index:
                contact_summary.loc[ind+(row,)] = np.nan
    contact_summary.sort_index(level=[0,1,2])


    ###################
    # flatten datasheets and combine
    # TODO: restructure this so that all of the datasheets and unstacked and then reorded based on shared level 0 columns before flattening
    ###################
    # org flattening
    org_final = org_summary.unstack(-1)
    for col in org_final.columns:
        if col[0].endswith(('_count', '_volume')):
            if col[2] not in col[0]:
                org_final.drop(col,axis=1, inplace=True)

    org_final.columns = ["_".join((col_name[-1], col_name[1], col_name[0])) for col_name in org_final.columns.to_flat_index()]

    #renaming, filling "NaN" with 0 when needed, and removing ER_std columns
    for col in org_final.columns:
    #     if '_count_in_' or '_fraction_in_' in col:
    #         org_final[col] = org_final[col].fillna(0)
        if col.endswith(("_count_volume","_sum_volume", "_mean_volume", "_median_volume")):
            org_final[col] = org_final[col].fillna(0)
        if col.endswith("_count_volume"):
            org_final.rename(columns={col:col.split("_")[0]+"_count"}, inplace=True)
        if col.startswith("ER_std_"):
            org_final.drop(columns=[col], inplace=True)
    org_final = org_final.reset_index()

    # contacts flattened
    contact_final = contact_summary.unstack(-1)
    contact_final.columns = ["_".join((col_name[-1], col_name[1], col_name[0])) for col_name in contact_final.columns.to_flat_index()]

    #renaming and filling "NaN" with 0 when needed
    for col in contact_final.columns:
        if "norm_to" in col:
            s = col.split("_")
            org_alpha = s[-2]
            cont_list = col.split("_")[0].split("X")

            if org_alpha.isupper():
                idx = ord(org_alpha)-ord('A')
                # print("upper: ", org_alpha, "-->", idx)
            elif org_alpha.islower():
                idx = ord(org_alpha)-ord('a')
                # print("lower: ", org_alpha, "-->", idx)
            else:
                ValueError("Naming scheme isn't correct for interaction normalization columns.")

            if idx < len(cont_list):
                org = cont_list[idx]
                s[-2] = org # replace letter with org name
                contact_final.rename(columns={col:"_".join(s)}, inplace=True)
            else:
                contact_final.drop(columns=[col], inplace=True)

    for col in contact_final.columns:
        if col.endswith(("_count_volume","_sum_volume", "_mean_volume", "_median_volume")):
            contact_final[col] = contact_final[col].fillna(0)
        if col.endswith("_count_volume"):
            contact_final.rename(columns={col:col.split("_")[0]+"_count"}, inplace=True)
    contact_final = contact_final.reset_index()

    # distributions flattened
    dist_final = dist_summary.unstack(-1)
    dist_final.columns = ["_".join((col_name[1], col_name[0])) for col_name in dist_final.columns.to_flat_index()]
    dist_final = dist_final.reset_index()

    # regions flattened & normalization added
    regions_final = regions_summary.unstack(-1)
    regions_final.columns = ["_".join((col_name[1], col_name[0])) for col_name in regions_final.columns.to_flat_index()]
    regions_final['nuc_volume_fraction'] = regions_final['nuc_volume'] / regions_final[f'{mask_name}_volume']
    regions_final = regions_final.reset_index()

    # # combining them all
    combined = pd.merge(org_final, contact_final, on=["dataset", "image_name"], how="outer")
    combined = pd.merge(combined, dist_final, on=["dataset", "image_name"], how="outer")
    combined = pd.merge(combined, regions_final, on=["dataset", "image_name"], how="outer").set_index(["dataset", "image_name"])
    combined.columns = [col.replace('sum', 'total') for col in combined.columns]

    ###################
    # export summary sheets
    # ###################
    org_summary.to_csv(out_path + f"/{out_preffix}per_org_summarystats.csv")
    contact_summary.to_csv(out_path + f"/{out_preffix}per_contact_summarystats.csv")
    dist_summary.to_csv(out_path + f"/{out_preffix}distribution_summarystats.csv")
    regions_summary.to_csv(out_path + f"/{out_preffix}per_region_summarystats.csv")
    combined.to_csv(out_path + f"/{out_preffix}summarystats_combined.csv")

    print(f"Processing of {fl_count} files from {ds_count} dataset(s) is complete.")
    return org_summary, contact_summary, dist_summary, regions_summary, combined #f"{fl_count} files from {ds_count} dataset(s) were processed"

In [ ]:
### FOR REFERENCE WHEN UPDATING 2.2 NOTEBOOK ###
def batch_summary_stats_20250805(csv_path_list: List[str],
                                out_path: str,
                                out_preffix: str,
                                mask_name: str):
    """" 
    csv_path_list: List[str],
        A list of path strings where .csv files to analyze are located.
    out_path: str,
        A path string where the summary data file will be output to
    out_preffix: str
        The prefix used to name the output file.    
    """
    # ds_count = 0
    # fl_count = 0
    # ###################
    # # Read in the csv files and combine them into one of each type
    # ###################
    # org_tabs = []
    # contact_tabs = []
    # dist_tabs = []
    # region_tabs = []

    # org = "_organelles"
    # contacts = "_interactions"
    # dist = "_distributions"
    # regions = "_regions"

    # for loc in csv_path_list:
    #     ds_count = ds_count + 1
    #     loc=Path(loc)
    #     files_store = sorted(loc.glob("*.csv"))
    #     for file in files_store:
    #         fl_count = fl_count + 1
    #         stem = file.stem

    #         if org in stem:
    #             test_orgs = pd.read_csv(file, index_col=0)
    #             test_orgs.insert(0, "dataset", stem[:-11])
    #             org_tabs.append(test_orgs)
    #         if contacts in stem:
    #             test_contact = pd.read_csv(file, index_col=0)
    #             test_contact.insert(0, "dataset", stem[:-13])
    #             contact_tabs.append(test_contact)
    #         if dist in stem:
    #             test_dist = pd.read_csv(file, index_col=0)
    #             test_dist.insert(0, "dataset", stem[:-14])
    #             dist_tabs.append(test_dist)
    #         if regions in stem:
    #             test_regions = pd.read_csv(file, index_col=0)
    #             test_regions.insert(0, "dataset", stem[:-8])
    #             region_tabs.append(test_regions)
            
    # org_df = pd.concat(org_tabs,axis=0, join='outer')
    # contacts_df = pd.concat(contact_tabs,axis=0, join='outer')
    # dist_df = pd.concat(dist_tabs,axis=0, join='outer')
    # regions_df = pd.concat(region_tabs,axis=0, join='outer')

    ###################
    # adding new metrics to the original sheets
    ###################    
    # org_list = org_df['object'].unique().tolist()
    # contact_cnt = batch_summary_interactions_2_20250805(contacts_df, 'X', org_list)

    # org_df = pd.merge(org_df, contact_cnt, on=['dataset', 'image_name', 'object', 'label'], how='left',sort=True)
    # org_df[contact_cnt.columns] = org_df[contact_cnt.columns].fillna(0)


    ###################
    # summary stat group
    ###################
    # group_by = ['dataset', 'image_name', 'object']
    # sharedcolumns = ["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"]
    # ag_func_standard = ['mean', 'median', 'std']

    # ###################
    # # summarize shared measurements between org_df and contacts_df
    # ###################
    # org_cont_tabs = []
    # for tab in [org_df, contacts_df]:
    #     tab1 = tab[group_by + ['volume']].groupby(group_by).agg(['count', 'sum'] + ag_func_standard)
    #     tab2 = tab[group_by + ['surface_area']].groupby(group_by).agg(['sum'] + ag_func_standard)
    #     tab3 = tab[group_by + sharedcolumns].groupby(group_by).agg(ag_func_standard)
    #     shared_metrics = pd.merge(tab1, tab2, 'outer', on=group_by)
    #     shared_metrics = pd.merge(shared_metrics, tab3, 'outer', on=group_by)
    #     org_cont_tabs.append(shared_metrics)

    # org_summary = org_cont_tabs[0]
    # contact_summary = org_cont_tabs[1]

    ###################
    # group metrics from regions_df similar to the above
    ###################
    # regions_summary = regions_df[group_by + ['volume', 'surface_area'] + sharedcolumns].set_index(group_by)
    # regions_summary.sort_index(axis=1, inplace=True)

    ###################
    # summarize extra metrics from org_df
    # ###################
    # columns2 = [col for col in org_df.columns if col.endswith(("_count", "_volume"))]
    # contact_counts_summary = org_df[group_by + columns2].groupby(group_by).agg(['sum'] + ag_func_standard)
    # org_summary = pd.merge(org_summary, contact_counts_summary, 'outer', on=group_by)


    ###################
    # summarize distribution measurements
    ###################
    # organelle distributions
    hist_dfs = []
    for ind in range(0,len(dist_df.index)):
        selection = dist_df.iloc[[ind]] #    selection = dist_df.loc[[ind]]
        bins_df = pd.DataFrame()
        wedges_df = pd.DataFrame()
        Z_df = pd.DataFrame()
        CV_df = pd.DataFrame()

        bins_df[['bins', 'masks', 'obj']] = selection[['XY_bins', 'XY_mask_vox_cnt_perbin', 'XY_obj_vox_cnt_perbin']]
        wedges_df[['bins', 'masks', 'obj']] = selection[['XY_wedges', 'XY_mask_vox_cnt_perwedge', 'XY_obj_vox_cnt_perwedge']]
        Z_df[['bins', 'masks', 'obj']] = selection[['Z_slices', 'Z_mask_vox_cnt', 'Z_obj_vox_cnt']]

        dfs = [selection[['dataset', 'image_name', 'object']].reset_index()]
        for df, prefix in zip([bins_df, wedges_df, Z_df], ["XY_bins_", "XY_wedges_", "Z_slices_"]):
            single_df = pd.DataFrame(list(zip(df["bins"].values[0][1:-1].split(", "), 
                                            df["obj"].values[0][1:-1].split(", "), 
                                            df["masks"].values[0][1:-1].split(", "))), columns =['bins', 'obj', 'mask']).astype(int)
            
            if "Z_" in prefix:
                single_df =  single_df.drop(single_df[single_df['mask'] == 0].index)
                single_df['bins'] = (single_df["bins"]/max(single_df.bins)*9.99).apply(np.floor)+1
                single_df = single_df.groupby("bins").agg(['sum']).reset_index()
                single_df.columns = ['bins',"obj","mask"]
        
            single_df['mask_fract'] = single_df['mask']/single_df['mask'].max()
            # single_df['obj_normed_tocell'] = (single_df["obj"]*single_df["mask_fract"]).fillna(0)
            single_df['obj_perc_per_bin'] = (single_df["obj"] / single_df["obj"].sum())*100
            single_df['obj_portion_normed_tobin'] = (single_df["obj_perc_per_bin"]/single_df["mask_fract"]).fillna(0)

            sumstats_df = pd.DataFrame()

            s = single_df['bins'].repeat(single_df['obj_portion_normed_tobin']*100)

            sumstats_df['hist_mean']=[s.mean()]
            sumstats_df['hist_median']=[s.median()]
            if single_df['obj_portion_normed_tobin'].sum() != 0: sumstats_df['hist_mode']=[s.mode().iloc[0]]
            else: sumstats_df['hist_mode']=['NaN']
            sumstats_df['hist_min']=[s.min()]
            sumstats_df['hist_max']=[s.max()]
            sumstats_df['hist_range']=[s.max() - s.min()]
            sumstats_df['hist_stdev']=[s.std()]
            sumstats_df['hist_skew']=[s.skew()]
            sumstats_df['hist_kurtosis']=[s.kurtosis()]
            sumstats_df['hist_var']=[s.var()]
            sumstats_df.columns = [prefix+col for col in sumstats_df.columns]
            dfs.append(sumstats_df.reset_index())

        CV_df = pd.DataFrame(list(zip(selection["XY_obj_cv_perbin"].values[0][1:-1].split(", "))), columns =['CV']).astype(float)
        sumstats_CV_df = pd.DataFrame()
        sumstats_CV_df['XY_bin_CV_mean'] = CV_df.mean()
        sumstats_CV_df['XY_bin_CV_median'] = CV_df.median()
        sumstats_CV_df['XY_bin_CV_std'] = CV_df.std()
        dfs.append(sumstats_CV_df.reset_index().drop(['index'], axis=1))

        combined_df = pd.concat(dfs, axis=1).drop(columns="index")
        hist_dfs.append(combined_df)
    dist_org_summary = pd.concat(hist_dfs, ignore_index=True)
    dist_org_summary

    # nucleus distribution
    nuc_dist_df = dist_df[["dataset", "image_name", 
                        "XY_bins", "XY_center_vox_cnt_perbin", "XY_mask_vox_cnt_perbin",
                        "XY_wedges", "XY_center_vox_cnt_perwedge", "XY_mask_vox_cnt_perwedge",
                        "Z_slices", "Z_center_vox_cnt", "Z_mask_vox_cnt"]].set_index(["dataset", "image_name"])
    nuc_hist_dfs = []
    for idx in nuc_dist_df.index.unique():
        selection = nuc_dist_df.loc[idx].iloc[[0]].reset_index()
        bins_df = pd.DataFrame()
        wedges_df = pd.DataFrame()
        Z_df = pd.DataFrame()

        bins_df[['bins', 'center', 'masks']] = selection[['XY_bins', 'XY_center_vox_cnt_perbin', 'XY_mask_vox_cnt_perbin']]
        wedges_df[['bins', 'center', 'masks']] = selection[['XY_wedges', 'XY_center_vox_cnt_perwedge', 'XY_mask_vox_cnt_perwedge']]
        Z_df[['bins', 'center', 'masks']] = selection[['Z_slices', 'Z_center_vox_cnt', 'Z_mask_vox_cnt']]

        dfs = [selection[['dataset', 'image_name']]]
        for df, prefix in zip([bins_df, wedges_df, Z_df], ["XY_bins_", "XY_wedges_", "Z_slices_"]):
            single_df = pd.DataFrame(list(zip(df["bins"].values[0][1:-1].split(", "), 
                                            df["masks"].values[0][1:-1].split(", "),
                                            df["center"].values[0][1:-1].split(", "))), columns =['bins', 'mask', 'obj']).astype(int)

            if "Z_" in prefix:
                single_df =  single_df.drop(single_df[single_df['mask'] == 0].index)
                single_df['bins'] = (single_df["bins"]/max(single_df.bins)*9.99).apply(np.floor)+1
                single_df = single_df.groupby("bins").agg(['sum']).reset_index()
                single_df.columns = ['bins',"mask","obj"]
        
            single_df['mask_fract'] = single_df['mask']/single_df['mask'].max()
            # single_df['obj_normed_tocell'] = (single_df["obj"]*single_df["mask_fract"]).fillna(0)
            single_df['obj_perc_per_bin'] = (single_df["obj"] / single_df["obj"].sum())*100
            single_df['obj_portion_normed_tobin'] = (single_df["obj_perc_per_bin"]/single_df["mask_fract"]).fillna(0)

            sumstats_df = pd.DataFrame()

            s = single_df['bins'].repeat(single_df['obj_portion_normed_tobin']*100)

            sumstats_df['hist_mean']=[s.mean()]
            sumstats_df['hist_median']=[s.median()]
            if single_df['obj_portion_normed_tobin'].sum() != 0: sumstats_df['hist_mode']=[s.mode().iloc[0]]
            else: sumstats_df['hist_mode']=['NaN']
            sumstats_df['hist_min']=[s.min()]
            sumstats_df['hist_max']=[s.max()]
            sumstats_df['hist_range']=[s.max() - s.min()]
            sumstats_df['hist_stdev']=[s.std()]
            sumstats_df['hist_skew']=[s.skew()]
            sumstats_df['hist_kurtosis']=[s.kurtosis()]
            sumstats_df['hist_var']=[s.var()]
            sumstats_df.columns = [prefix+col for col in sumstats_df.columns]
            dfs.append(sumstats_df.reset_index())
        combined_df = pd.concat(dfs, axis=1).drop(columns="index")
        nuc_hist_dfs.append(combined_df)
    dist_center_summary = pd.concat(nuc_hist_dfs, ignore_index=True)
    dist_center_summary.insert(2, column="object", value="nuc")

    dist_summary = pd.concat([dist_org_summary, dist_center_summary], axis=0).set_index(group_by).sort_index()
    dist_summary.sort_index(axis=1, inplace=True)


    ###################
    # add normalization
    ###################
    # organelle area fraction
    area_fractions = []
    for idx in org_summary.index.unique():
        org_vol = org_summary.loc[idx][('volume', 'sum')]
        cell_vol = regions_summary.loc[idx[:-1] + (mask_name,)]["volume"]
        afrac = org_vol/cell_vol
        area_fractions.append(afrac)
    org_summary[('volume', 'fraction')] = area_fractions
    org_summary.sort_index(axis=1, inplace=True)

    ### ONLY IN COMBO ANALYSIS
    # number and area of individuals organelle involved in contact
    # contact_summary = normalize_interaction_volumes_20250805(contact_summary, org_summary)

    # all_pos = all_combo_20250804(org_list, "X")
    # for ind in contact_summary.index.droplevel(2).unique().to_list():
    #     for row in all_pos:
    #         if ind+(row,) not in contact_summary.index:
    #             contact_summary.loc[ind+(row,)] = np.nan
    # contact_summary.sort_index(level=[0,1,2])


    ###################
    # flatten datasheets and combine
    # TODO: restructure this so that all of the datasheets and unstacked and then reorded based on shared level 0 columns before flattening
    ###################
    # org flattening
    org_final = org_summary.unstack(-1)
    for col in org_final.columns:
        if col[0].endswith(('_count', '_volume')):
            if col[2] not in col[0]:
                org_final.drop(col,axis=1, inplace=True)

    org_final.columns = ["_".join((col_name[-1], col_name[1], col_name[0])) for col_name in org_final.columns.to_flat_index()]

    #renaming, filling "NaN" with 0 when needed, and removing ER_std columns
    for col in org_final.columns:
    #     if '_count_in_' or '_fraction_in_' in col:
    #         org_final[col] = org_final[col].fillna(0)
        if col.endswith(("_count_volume","_sum_volume", "_mean_volume", "_median_volume")):
            org_final[col] = org_final[col].fillna(0)
        if col.endswith("_count_volume"):
            org_final.rename(columns={col:col.split("_")[0]+"_count"}, inplace=True)
        if col.startswith("ER_std_"):
            org_final.drop(columns=[col], inplace=True)
    org_final = org_final.reset_index()

    # contacts flattened
    contact_final = contact_summary.unstack(-1)
    contact_final.columns = ["_".join((col_name[-1], col_name[1], col_name[0])) for col_name in contact_final.columns.to_flat_index()]

    #renaming and filling "NaN" with 0 when needed
    for col in contact_final.columns:
        if "norm_to" in col:
            s = col.split("_")
            org_alpha = s[-2]
            cont_list = col.split("_")[0].split("X")

            if org_alpha.isupper():
                idx = ord(org_alpha)-ord('A')
                # print("upper: ", org_alpha, "-->", idx)
            elif org_alpha.islower():
                idx = ord(org_alpha)-ord('a')
                # print("lower: ", org_alpha, "-->", idx)
            else:
                ValueError("Naming scheme isn't correct for interaction normalization columns.")

            if idx < len(cont_list):
                org = cont_list[idx]
                s[-2] = org # replace letter with org name
                contact_final.rename(columns={col:"_".join(s)}, inplace=True)
            else:
                contact_final.drop(columns=[col], inplace=True)

    for col in contact_final.columns:
        if col.endswith(("_count_volume","_sum_volume", "_mean_volume", "_median_volume")):
            contact_final[col] = contact_final[col].fillna(0)
        if col.endswith("_count_volume"):
            contact_final.rename(columns={col:col.split("_")[0]+"_count"}, inplace=True)
    contact_final = contact_final.reset_index()

    # distributions flattened
    dist_final = dist_summary.unstack(-1)
    dist_final.columns = ["_".join((col_name[1], col_name[0])) for col_name in dist_final.columns.to_flat_index()]
    dist_final = dist_final.reset_index()

    # regions flattened & normalization added
    regions_final = regions_summary.unstack(-1)
    regions_final.columns = ["_".join((col_name[1], col_name[0])) for col_name in regions_final.columns.to_flat_index()]
    regions_final['nuc_volume_fraction'] = regions_final['nuc_volume'] / regions_final[f'{mask_name}_volume']
    regions_final = regions_final.reset_index()

    # # combining them all
    combined = pd.merge(org_final, contact_final, on=["dataset", "image_name"], how="outer")
    combined = pd.merge(combined, dist_final, on=["dataset", "image_name"], how="outer")
    combined = pd.merge(combined, regions_final, on=["dataset", "image_name"], how="outer").set_index(["dataset", "image_name"])
    combined.columns = [col.replace('sum', 'total') for col in combined.columns]

    ###################
    # export summary sheets
    # ###################
    org_summary.to_csv(out_path + f"/{out_preffix}per_org_summarystats.csv")
    contact_summary.to_csv(out_path + f"/{out_preffix}per_contact_summarystats.csv")
    dist_summary.to_csv(out_path + f"/{out_preffix}distribution_summarystats.csv")
    regions_summary.to_csv(out_path + f"/{out_preffix}per_region_summarystats.csv")
    combined.to_csv(out_path + f"/{out_preffix}summarystats_combined.csv")

    print(f"Processing of {fl_count} files from {ds_count} dataset(s) is complete.")
    return org_summary, contact_summary, dist_summary, regions_summary, combined #f"{fl_count} files from

# **BATCH PROCESS QUANTIFICATION**

Input information about your data in the block below then run the analysis in the subsequent block.

In [41]:
#### USER REQUIRED INPUTS ###
quant_seg_path="Z:/Cohen Lab/Maria Clara/2_Lab data/9_Napari/Segmentation iNday7/FINAL iNd7"
quant_out_path="Z:/Cohen Lab/Maria Clara/2_Lab data/9_Napari/OUTPUT new code/iN day7/soma"
quant_raw_path="Z:/Cohen Lab/Maria Clara/2_Lab data/1_Multispectral data/2023/112023_MSi08-L_Neuronal Differentiation - iPSCs-hNGN2/Deconvolved iN day7 images 052024 water Cp/tiff/processed images/done"


quant_raw_file_type = ".tiff"
quant_organelle_names = ['LD', 'ER', 'golgi', 'lyso', 'mito', 'perox']       # list the organelle file suffixes for all the objects you want to include in your analysis
quant_organelle_channels= [0, 6, 4, 2, 3, 5]                                 # list of the intensity channel indexes associated to your organelle list
quant_region_names = ['soma', 'nuc']                                         # list the mask (e.g., 'cell') and distribution centering object (e.g., 'nuc') file suffixes 
quant_masks_file_name= quant_region_names                                     # list the cell and nucleus file suffixes again -- this duplicate is included for the v2.0 version of the analysis and not removed for simplicity here (NOTE: it does NOT run subregion analysis)
quant_mask = 'soma'                                                          # the file suffix name associated to the mask you want to use in your analysis

In [ ]:
out = QC_mask_quick(out_file_name = "08102025_iNday7_soma",
                    seg_path=quant_seg_path,
                    out_path=quant_out_path, 
                    raw_path=quant_raw_path, 
                    raw_file_type=quant_raw_file_type,
                    organelle_names=quant_organelle_names,
                    organelle_channels=quant_organelle_channels,
                    region_names=quant_region_names,
                    masks_file_name=quant_masks_file_name,
                    mask=quant_mask,
                    scale=True,
                    seg_suffix='-',
                    include_org_morph = True,
                    include_region_morph = True,
                    include_interactions = True,
                    include_distribution = True,
                    include_interact_dist = True,
                    dist_centering_obj = 'nuc', 
                    dist_num_bins = 5,
                    dist_center_on = False,
                    dist_keep_center_as_bin = True,
                    dist_zernike_degrees = None)

Beginning check for: Z:\Cohen Lab\Maria Clara\2_Lab data\1_Multispectral data\2023\112023_MSi08-L_Neuronal Differentiation - iPSCs-hNGN2\Deconvolved iN day7 images 052024 water Cp\tiff\processed images\done\01252024_MSi08L_iN_Day7_BR5_N01_Unmixing_0_cmle.ome.tiff
Beginning check for: Z:\Cohen Lab\Maria Clara\2_Lab data\1_Multispectral data\2023\112023_MSi08-L_Neuronal Differentiation - iPSCs-hNGN2\Deconvolved iN day7 images 052024 water Cp\tiff\processed images\done\01252024_MSi08L_iN_Day7_BR5_N02_Unmixing_0_cmle.ome.tiff
Beginning check for: Z:\Cohen Lab\Maria Clara\2_Lab data\1_Multispectral data\2023\112023_MSi08-L_Neuronal Differentiation - iPSCs-hNGN2\Deconvolved iN day7 images 052024 water Cp\tiff\processed images\done\01252024_MSi08L_iN_Day7_BR5_N03_Unmixing.czi_noGolgi_0_cmle.ome.tiff
Beginning check for: Z:\Cohen Lab\Maria Clara\2_Lab data\1_Multispectral data\2023\112023_MSi08-L_Neuronal Differentiation - iPSCs-hNGN2\Deconvolved iN day7 images 052024 water Cp\tiff\processed i

In [ ]:
#### RUN ANALYSIS ### 
count = batch_process_quantification_20250804(out_file_name = "08102025_iNday7_soma",
                                                seg_path=quant_seg_path,
                                                out_path=quant_out_path, 
                                                raw_path=quant_raw_path, 
                                                raw_file_type=quant_raw_file_type,
                                                organelle_names=quant_organelle_names,
                                                organelle_channels=quant_organelle_channels,
                                                region_names=quant_region_names,
                                                masks_file_name=quant_masks_file_name,
                                                mask=quant_mask,
                                                scale=True,
                                                seg_suffix='-',
                                                include_org_morph = True,
                                                include_region_morph = True,
                                                include_interactions = True,
                                                include_distribution = True,
                                                include_interact_dist = True,
                                                dist_centering_obj = 'nuc', 
                                                dist_num_bins = 5,
                                                dist_center_on = False,
                                                dist_keep_center_as_bin = True,
                                                dist_zernike_degrees = None)

### **Repeat the analysis above for all experimental replicates in your dataset.**

# **BATCH PROCESS SUMMARY STATS (per cell)**

Input information about your data in the block below then run the analysis in the subsequent block.

**NOTE:** *If you are running batch_process_quantification() function multiple times using* **different masks** *(e.g., subregion analysis), the batch_summary_stats() function below should be run for each different mask type separately.*

In [11]:
#### USER REQUIRED INPUTS ###
# A list of file paths to the data you wish to include in this analysis - include all experimental replicates together here.
# If you use a different masks to analyze the data in different subregions, run the data per mask type separately.
sumstat_csv_path_list = [quant_out_path]

# location to save summary stats output files
sumstat_out_path = quant_out_path

# output file name prefix used during saving output files
sumstat_out_preffix = "iNday7_soma_08102025_SumStat"

In [15]:
#### RUN ANALYSIS ### 
out = batch_summary_stats_20250805(csv_path_list=sumstat_csv_path_list, 
                                   out_path=sumstat_out_path,
                                   out_preffix=sumstat_out_preffix,
                                   mask_name='soma')

C:\Users\Cohen Workstation 3\AppData\Local\Temp\ipykernel_14864\2545399131.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  interaction_cnt[[f"org{cha}" for cha in string.ascii_uppercase[:(len(max(interaction_cnt["object"].str.split(splitter), key=len)))]]] = interaction_cnt["object"].str.split(splitter, expand=True)
C:\Users\Cohen Workstation 3\AppData\Local\Temp\ipykernel_14864\2545399131.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  interaction_cnt[[f"org{cha}" for cha in string.ascii_uppercas

Processing of 4 files from 1 dataset(s) is complete.
